# 02 Launch, Starshield and AI Models

Launch counts only external revenue. Internal Starlink launches are an
avoided Starlink capex cost, not launch revenue. AI is split between
contracted terrestrial GPU rental economics and speculative orbital AI.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from valuation import (
    MarketPremiumInputs,
    OptionScenario,
    SegmentValuation,
    SotpInputs,
    build_sensitivity_grid,
    market_premium_value,
    probability_weighted_option_value,
    sotp_equity_value,
)

DATA_DIR = next(
    candidate / "apps/notebooks/studies/spacex_ipo/data"
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "apps/notebooks/studies/spacex_ipo/data").exists()
)
USD_BN = 1_000_000_000
pd.options.display.float_format = "{:,.2f}".format

In [ ]:
assumptions = pd.read_csv(DATA_DIR / "segment_assumptions.csv")
cases = ["bear", "base", "bull"]
segment_tables = {}
for segment in ["Launch", "Starshield", "AI"]:
    table = (
        assumptions[assumptions["segment"].eq(segment)]
        .pivot_table(index="case", columns="metric", values="value", aggfunc="first")
        .reindex(cases)
    )
    segment_tables[segment] = table
segment_tables["Launch"]

In [ ]:
launch = segment_tables["Launch"].assign(
    ebitda_usd_bn=lambda df: df["2030 revenue"] * df["EBITDA margin"],
    ev_revenue_multiple=[5.0, 7.5, 10.0],
    ev_ebitda_multiple=[17.0, 22.0, 28.0],
)
launch["selected_value_usd_bn"] = (
    launch["2030 revenue"] * launch["ev_revenue_multiple"]
    + launch["ebitda_usd_bn"] * launch["ev_ebitda_multiple"]
) / 2
launch

In [ ]:
starshield = segment_tables["Starshield"].assign(
    defense_multiple=[8.0, 12.0, 16.0],
    concentration_haircut=[0.30, 0.20, 0.10],
)
starshield["gross_value_usd_bn"] = starshield["2030 revenue"] * starshield["defense_multiple"]
starshield["selected_value_usd_bn"] = starshield["gross_value_usd_bn"] * (
    1 - starshield["concentration_haircut"]
)
starshield

In [ ]:
ai = segment_tables["AI"].assign(
    ebitda_usd_bn=lambda df: df["2030 revenue"] * df["EBITDA margin"],
    ev_revenue_multiple=[6.0, 10.0, 15.0],
    contract_duration_haircut=[0.35, 0.20, 0.10],
)
ai["gross_value_usd_bn"] = ai["2030 revenue"] * ai["ev_revenue_multiple"]
ai["selected_value_usd_bn"] = ai["gross_value_usd_bn"] * (1 - ai["contract_duration_haircut"])
ai

In [ ]:
ai_sensitivity = build_sensitivity_grid(
    row_values=[15, 30, 45, 60, 75],
    column_values=[0.10, 0.20, 0.30, 0.40],
    formula=lambda revenue, gross_margin: revenue * gross_margin,
    row_name="AI revenue (USD bn)",
    column_name="AI gross margin",
)
ai_sensitivity